# AI & 기계학습 방법론 — 실습 따라하기SSAFY 강의 4차시(선형회귀 / 로지스틱회귀 / 신경망 모델 / 신경망 적합)를 **코드로 직접 확인**하는 노트북입니다.* 필요한 라이브러리: `numpy`, `matplotlib` (선택: `scikit-learn` — 없어도 전부 동작합니다)* 위에서부터 순서대로 실행하세요. 각 셀은 앞 셀의 결과를 사용합니다.| 목차 | 내용 ||---|---|| 0 | 준비 || 1 | 단순선형회귀 · 최소제곱법 · RSS · R² || 2 | 다중선형회귀 · 정규방정식 · 다중공선성 || 3 | 로지스틱회귀 · 시그모이드 · 오즈/로짓 · MLE || 4 | Shallow 네트워크 · ReLU · 조각별 선형 · 보편적 근사 || 5 | Deep 네트워크 · 접기(folding) · 행렬 표기 || 6 | 손실함수 · 경사하강법 · 학습률 || 7 | SGD (확률적 경사하강법) || 8 | 역전파(Backpropagation) 직접 구현 + 수치미분 검증 |

## 0. 준비

In [ ]:
import numpy as npimport matplotlibimport matplotlib.pyplot as plt# 한글 폰트(있으면 사용, 없으면 무시 — 그래프는 영어 라벨로 그립니다)for f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:    try:        matplotlib.rc("font", family=f); break    except Exception:        passmatplotlib.rcParams["axes.unicode_minus"] = Falserng = np.random.default_rng(0)print("numpy", np.__version__)

---## 1. 단순선형회귀 (Simple Linear Regression)$$Y=\beta_0+\beta_1X+\varepsilon$$강의의 광고 데이터(TV 광고비 → 매출)와 비슷한 데이터를 만들어 봅니다.실제 강의 자료의 추정 결과는 $\hat\beta_0=7.0325,\ \hat\beta_1=0.0475,\ R^2=0.612$ 였습니다.

In [ ]:
# TV 광고비(0~300, 단위 1백만원) → Salesn = 200TV = rng.uniform(0, 300, n)sales = 7.0325 + 0.0475*TV + rng.normal(0, 3.2, n)   # 참값에 노이즈plt.figure(figsize=(5,3.5))plt.scatter(TV, sales, s=12, c="crimson", alpha=.6)plt.xlabel("TV"); plt.ylabel("Sales"); plt.title("Advertising data (simulated)")plt.tight_layout(); plt.show()

### 1-1. 최소제곱법 closed-form 해$$\hat\beta_1=\frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2},\qquad \hat\beta_0=\bar y-\hat\beta_1\bar x$$

In [ ]:
def ols_simple(x, y):    xbar, ybar = x.mean(), y.mean()    b1 = ((x-xbar)*(y-ybar)).sum() / ((x-xbar)**2).sum()    b0 = ybar - b1*xbar    return b0, b1b0, b1 = ols_simple(TV, sales)print(f"beta0_hat = {b0:.4f}")print(f"beta1_hat = {b1:.5f}")print("해석: TV 광고비 1단위(1백만원) 증가 시 매출 약", round(b1*100, 2), "만원 증가")

### 1-2. 잔차(residual), RSS, R²

In [ ]:
y_hat = b0 + b1*TVresid = sales - y_hat              # e_i = y_i - y_hat_iRSS   = (resid**2).sum()           # 잔차제곱합TSS   = ((sales - sales.mean())**2).sum()R2    = 1 - RSS/TSSprint(f"RSS = {RSS:.2f}")print(f"TSS = {TSS:.2f}")print(f"R^2 = {R2:.4f}   (= 매출 변동의 {R2*100:.1f}% 를 TV 광고비로 설명)")

In [ ]:
# 잔차 그림 (강의 <그림6>, <그림7>과 동일한 아이디어)plt.figure(figsize=(5.5,3.8))plt.scatter(TV, sales, s=12, c="crimson", zorder=3)order = np.argsort(TV)plt.plot(TV[order], y_hat[order], c="blue", lw=2, zorder=4, label="OLS line")for xi, yi, yh in zip(TV[::4], sales[::4], y_hat[::4]):    plt.plot([xi, xi], [yi, yh], c="gray", lw=.8, zorder=2)plt.legend(); plt.xlabel("TV"); plt.ylabel("Sales"); plt.title("residuals")plt.tight_layout(); plt.show()

### 1-3. RSS는 정말 최소일까? (직접 확인)$\hat\beta_1$ 주변을 훑으며 RSS를 그려 봅니다. closed-form 해에서 최소가 되는지 눈으로 확인하세요.

In [ ]:
grid = np.linspace(b1-0.02, b1+0.02, 200)rss_curve = [((sales - (b0 + g*TV))**2).sum() for g in grid]plt.figure(figsize=(5,3.2))plt.plot(grid, rss_curve)plt.axvline(b1, c="red", ls="--", label=f"closed-form b1={b1:.5f}")plt.xlabel("beta1"); plt.ylabel("RSS"); plt.legend()plt.tight_layout(); plt.show()

### 1-4. 표준오차 · t통계량 · p-value강의 표의 Std.Error / t-statistic / p-value 를 직접 계산해 봅니다.$$\mathrm{SE}(\hat\beta_1)=\sqrt{\frac{\sigma^2}{\sum(x_i-\bar x)^2}},\qquad t=\frac{\hat\beta_1}{\mathrm{SE}(\hat\beta_1)}$$

In [ ]:
nn = len(TV)sigma2 = RSS/(nn-2)                       # 잔차 분산 추정Sxx = ((TV-TV.mean())**2).sum()se_b1 = np.sqrt(sigma2/Sxx)se_b0 = np.sqrt(sigma2*(1/nn + TV.mean()**2/Sxx))t0, t1 = b0/se_b0, b1/se_b1print(f"{'':12s}{'Coefficient':>12s}{'Std.Error':>12s}{'t-stat':>10s}")print(f"{'Intercept':12s}{b0:12.4f}{se_b0:12.4f}{t0:10.2f}")print(f"{'TV':12s}{b1:12.4f}{se_b1:12.4f}{t1:10.2f}")print()print("t의 절대값이 매우 크므로 p-value는 사실상 0 → 통계적으로 유의")

---## 2. 다중선형회귀 (Multiple Linear Regression)$$Y=\beta_0+\beta_1X_1+\cdots+\beta_pX_p+\varepsilon,\qquad \hat{\boldsymbol\beta}=(\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y$$강의 결과: TV 0.046, radio 0.189, newspaper -0.001(유의하지 않음), $R^2=0.897$

In [ ]:
radio     = rng.uniform(0, 50, n)newspaper = rng.uniform(0, 110, n)# newspaper는 진짜로는 영향이 없다(계수 0)sales2 = 2.939 + 0.046*TV + 0.189*radio + 0.0*newspaper + rng.normal(0, 1.7, n)X = np.column_stack([np.ones(n), TV, radio, newspaper])   # 1열은 절편beta = np.linalg.inv(X.T @ X) @ X.T @ sales2              # 정규방정식names = ["Intercept", "TV", "radio", "newspaper"]resid2 = sales2 - X @ betaRSS2 = (resid2**2).sum()sigma2_2 = RSS2/(n - X.shape[1])cov = sigma2_2 * np.linalg.inv(X.T @ X)se = np.sqrt(np.diag(cov))print(f"{'':12s}{'Coefficient':>12s}{'Std.Error':>12s}{'t-stat':>10s}")for nm, b, s in zip(names, beta, se):    print(f"{nm:12s}{b:12.4f}{s:12.4f}{b/s:10.2f}")R2m = 1 - RSS2/((sales2-sales2.mean())**2).sum()print(f"\nR^2 = {R2m:.4f}")print("→ newspaper의 t값만 작다 = 유의하지 않다 (강의 결과와 동일한 패턴)")

### 2-1. 다중공선성(multicollinearity) 실험두 입력의 상관을 높이면, **진짜 계수는 그대로인데 추정치가 요동친다**는 것을 확인합니다.

In [ ]:
def experiment(rho, trials=200, m=50):    b1s, b2s = [], []    for _ in range(trials):        a = rng.normal(size=m)        b = rng.normal(size=m)        x1 = a        x2 = rho*a + np.sqrt(max(0, 1-rho**2))*b        y  = 1 + 2*x1 + 3*x2 + rng.normal(0, 1.0, m)   # 진짜 계수 (2, 3)        Xm = np.column_stack([np.ones(m), x1, x2])        bb = np.linalg.solve(Xm.T@Xm, Xm.T@y)        b1s.append(bb[1]); b2s.append(bb[2])    return np.array(b1s), np.array(b2s)for rho in [0.0, 0.9, 0.99]:    b1s, b2s = experiment(rho)    print(f"corr={rho:4.2f} | b1 평균 {b1s.mean():6.3f} 표준편차 {b1s.std():6.3f}"          f" | b2 평균 {b2s.mean():6.3f} 표준편차 {b2s.std():6.3f}")print("\n→ 평균은 진짜 값(2, 3)에 가깝지만, 상관이 높을수록 '흔들림(표준편차)'이 폭증")

### 2-2. 상관 ≠ 인과강의 예시: “아이스크림 소비량(X)”과 “상어에 물리는 사건(Y)”.숨은 변수(기온)가 둘 다를 끌어올리면 X와 Y는 강한 상관을 보이지만 인과 관계는 없습니다.

In [ ]:
temp      = rng.uniform(0, 35, 300)          # 숨은 변수: 기온icecream  = 2*temp + rng.normal(0, 8, 300)shark     = 0.3*temp + rng.normal(0, 2, 300)  # 아이스크림과 직접 관계 없음!r = np.corrcoef(icecream, shark)[0,1]print(f"corr(아이스크림, 상어사고) = {r:.3f}  ← 매우 높다")# 기온을 통제(회귀에 포함)하면?Xc = np.column_stack([np.ones(300), icecream, temp])bc = np.linalg.solve(Xc.T@Xc, Xc.T@shark)print(f"기온을 함께 넣었을 때 아이스크림 계수 = {bc[1]:.4f}  ← 거의 0")

---## 3. 로지스틱회귀 (Logistic Regression)$$p(X;\beta)=\frac{1}{1+e^{-(\beta_0+\beta_1X)}},\qquad \log\frac{p}{1-p}=\beta_0+\beta_1X$$강의 결과(신용카드 연체): Intercept $-10.6513$, balance $0.0055$

In [ ]:
def sigmoid(z):    return 1/(1+np.exp(-z))z = np.linspace(-8, 8, 400)plt.figure(figsize=(5,3))plt.plot(z, sigmoid(z))plt.axhline(0, c="gray", lw=.7); plt.axhline(1, c="gray", lw=.7)plt.axhline(.5, c="orange", ls="--", lw=.8)plt.title("sigmoid"); plt.xlabel("z"); plt.ylabel("y")plt.tight_layout(); plt.show()print("z=+inf → ", sigmoid(50), " / z=-inf → ", sigmoid(-50))

### 3-1. 왜 선형회귀는 분류에 부적절한가

In [ ]:
# 이진 라벨 데이터bal = rng.uniform(0, 2600, 400)p_true = sigmoid(-10.6513 + 0.0055*bal)default = (rng.random(400) < p_true).astype(float)# (부적절한) 선형회귀Xb = np.column_stack([np.ones(400), bal])blin = np.linalg.solve(Xb.T@Xb, Xb.T@default)xs = np.linspace(0, 2600, 200)plt.figure(figsize=(5.5,3.4))plt.scatter(bal, default, s=8, alpha=.35, c="steelblue")plt.plot(xs, blin[0]+blin[1]*xs, c="crimson", label="linear regression")plt.plot(xs, sigmoid(-10.6513+0.0055*xs), c="green", lw=2, label="logistic")plt.axhline(0, c="gray", lw=.6); plt.axhline(1, c="gray", lw=.6)plt.legend(); plt.xlabel("Balance"); plt.ylabel("P(Default)")plt.tight_layout(); plt.show()print("선형회귀 예측값 범위:", round((blin[0]+blin[1]*xs).min(),3), "~", round((blin[0]+blin[1]*xs).max(),3))print("→ 0보다 작거나 1보다 큰 값이 나옴 = 확률로 쓸 수 없음")

### 3-2. MLE로 직접 학습 (log-likelihood 최대화 = 음의 로그우도 최소화)$$\log\mathcal L(\beta)=\sum_i\bigl[y_i\log p_i+(1-y_i)\log(1-p_i)\bigr]$$경사상승/하강으로 풀어 봅니다. (기울기: $\sum_i (p_i-y_i)x_i$)

In [ ]:
def fit_logistic(X, y, lr=0.1, iters=8000):    b = np.zeros(X.shape[1])    hist = []    for t in range(iters):        p = sigmoid(X @ b)        grad = X.T @ (p - y) / len(y)      # 음의 로그우도의 기울기        b -= lr*grad        if t % 100 == 0:            ll = np.sum(y*np.log(p+1e-12) + (1-y)*np.log(1-p+1e-12))            hist.append(ll)    return b, hist# 스케일 차이가 크면 수렴이 느리므로 balance를 1000으로 나눠 학습 후 되돌립니다Xs = np.column_stack([np.ones(400), bal/1000])bhat_s, hist = fit_logistic(Xs, default, lr=0.5, iters=60000)b0_log, b1_log = bhat_s[0], bhat_s[1]/1000print(f"추정 Intercept = {b0_log:.4f}   (강의: -10.6513)")print(f"추정 balance   = {b1_log:.6f}  (강의:  0.0055)")plt.figure(figsize=(5,2.8)); plt.plot(hist)plt.xlabel("iteration (x100)"); plt.ylabel("log-likelihood"); plt.title("MLE: log L 증가")plt.tight_layout(); plt.show()

### 3-3. 오즈(odds)와 로짓(logit) — “2배 늘려도 확률은 97배”

In [ ]:
for x in [1000, 2000]:    z = -10.6513 + 0.0055*x    p = sigmoid(z)    print(f"X={x:5d} | z(logit)={z:8.4f} | p={p:.4f} | odds={p/(1-p):.4f}")p1 = sigmoid(-10.6513+0.0055*1000); p2 = sigmoid(-10.6513+0.0055*2000)print(f"\n입력 2배 → 확률 {p2/p1:.1f}배 (강의: 약 97배)")print("로짓은 선형이지만, 확률은 선형이 아니다!")

---## 4. Shallow 네트워크$$y=\phi_0+\phi_1\mathrm a[\theta_{10}+\theta_{11}x]+\phi_2\mathrm a[\theta_{20}+\theta_{21}x]+\phi_3\mathrm a[\theta_{30}+\theta_{31}x]$$

In [ ]:
def relu(z):    return np.maximum(0, z)theta = np.array([[-0.2, 0.9],     # theta_10, theta_11                  [-0.9, 1.2],                  [ 1.1,-0.7]])phi   = np.array([0.0, -1.0, 1.0, 0.8])   # phi_0, phi_1, phi_2, phi_3def shallow(x):    h = relu(theta[:,0][:,None] + theta[:,1][:,None]*x)   # (3, N)    return phi[0] + phi[1:] @ h, hx = np.linspace(0, 2, 500)y, h = shallow(x)fig, ax = plt.subplots(2, 3, figsize=(10,4.6), sharex=True)for d in range(3):    ax[0,d].plot(x, theta[d,0]+theta[d,1]*x, c="tab:orange")    ax[0,d].set_title(f"linear: th{d+1}0 + th{d+1}1*x", fontsize=9)    ax[1,d].plot(x, h[d], c="tab:green")    ax[1,d].set_title(f"h{d+1} = ReLU[...]", fontsize=9)for a in ax.ravel(): a.axhline(0, c="gray", lw=.6)plt.tight_layout(); plt.show()plt.figure(figsize=(5,3))plt.plot(x, y, lw=2)for d in range(3):    if theta[d,1] != 0:        k = -theta[d,0]/theta[d,1]        if 0 < k < 2: plt.axvline(k, ls="--", c="wheat")plt.title("y = phi0 + sum phi_d h_d  (piecewise linear)")plt.xlabel("x"); plt.ylabel("y"); plt.tight_layout(); plt.show()print("꺾이는 지점 =", [round(float(-theta[d,0]/theta[d,1]),3) for d in range(3)])print("→ 조각(linear region) 수 = 구간 안의 꺾임 수 + 1")

### 4-1. 보편적 근사 정리 확인Hidden unit 수 $D$를 늘리면서 목표 함수를 근사합니다. 조각 수는 $D+1$.

In [ ]:
def approximate(g, D, a=0, b=2):    knots = np.linspace(a, b, D+2)[1:-1]           # D개의 꺾임점    pts = np.concatenate([[a], knots, [b]])    slopes = np.diff(g(pts))/np.diff(pts)    coef = np.concatenate([[slopes[0]], np.diff(slopes)])   # phi들    def f(x):        y = g(a) + coef[0]*(x-a)        for k, c in zip(knots, coef[1:]):            y = y + c*relu(x-k)        return y    return fg = lambda t: np.sin(2.2*t)xs = np.linspace(0, 2, 800)plt.figure(figsize=(9,2.8))for i, D in enumerate([4, 9, 19]):    f = approximate(g, D)    mse = np.mean((f(xs)-g(xs))**2)    plt.subplot(1,3,i+1)    plt.plot(xs, g(xs), "k--", lw=1)    plt.plot(xs, f(xs), lw=2)    plt.title(f"D={D} → {D+1} regions\nMSE={mse:.5f}", fontsize=9)plt.tight_layout(); plt.show()

---## 5. Deep 네트워크 — 접기(folding)$$\mathbf h_1=\mathrm a[\boldsymbol\beta_0+\boldsymbol\Omega_0\mathbf x],\quad\mathbf h_2=\mathrm a[\boldsymbol\beta_1+\boldsymbol\Omega_1\mathbf h_1],\quad\mathbf y=\boldsymbol\beta_2+\boldsymbol\Omega_2\mathbf h_2$$

In [ ]:
def count_regions(f, a=-1, b=1, N=20000):    xs = np.linspace(a, b, N)    ys = f(xs)    slopes = np.diff(ys)/np.diff(xs)    changes = np.sum(np.abs(np.diff(slopes)) > 1e-7)    return changes + 1rg = np.random.default_rng(3)def rand_layer(nin, nout, r):    return r.normal(0, 1.4, (nout, nin)), r.normal(0, .7, nout)# Deep: 1 -> 3 -> 1 -> 3 -> 1  (은닉층 2개, 각 3유닛)  파라미터 20개W0,b0d = rand_layer(1,3,rg); W1,b1d = rand_layer(3,1,rg)W2,b2d = rand_layer(1,3,rg); W3,b3d = rand_layer(3,1,rg)def deep(x):    x = np.atleast_1d(x)[None,:]    h = relu(W0@x + b0d[:,None]); y = W1@h + b1d[:,None]    h2= relu(W2@y + b2d[:,None]); y2= W3@h2 + b3d[:,None]    return y2.ravel()# Shallow: 1 -> 6 -> 1  파라미터 19개V0,c0 = rand_layer(1,6,rg); V1,c1 = rand_layer(6,1,rg)def shallow6(x):    x = np.atleast_1d(x)[None,:]    return (V1@relu(V0@x + c0[:,None]) + c1[:,None]).ravel()xs = np.linspace(-1,1,2000)plt.figure(figsize=(9,3))plt.subplot(1,2,1); plt.plot(xs, shallow6(xs)); plt.title(f"Shallow(6 units, 19 params)\nregions={count_regions(shallow6)}")plt.subplot(1,2,2); plt.plot(xs, deep(xs), c="tab:blue"); plt.title(f"Deep(3+3 units, 20 params)\nregions={count_regions(deep)}")plt.tight_layout(); plt.show()print("강의: shallow 최대 7구역 / deep 최대 16구역 → Deep의 표현력이 높다")

### 5-1. “접기” 직관 — 1층 출력이 접히면 2층이 그린 그림이 반복된다

In [ ]:
def f1(x):    # 1층: V자/A자 모양 접기    x = np.atleast_1d(x)    return -1 + 2*relu(x+0.5) - 4*relu(x) + 2*relu(x-0.5)*2def f2(y):    # 2층: y를 입력으로 받는 간단한 꺾은선    return relu(y+0.5) - 2*relu(y)xs = np.linspace(-1,1,1000)fig, ax = plt.subplots(1,3, figsize=(10,2.8))ax[0].plot(xs, f1(xs)); ax[0].set_title("b) y = f1(x)  (접기)")ys = np.linspace(f1(xs).min(), f1(xs).max(), 400)ax[1].plot(ys, f2(ys), c="tab:orange"); ax[1].set_title("c) y' = f2(y)")ax[2].plot(xs, f2(f1(xs)), c="tab:green"); ax[2].set_title("d) y' = f2(f1(x))  (반복 패턴!)")plt.tight_layout(); plt.show()

---## 6. 손실함수와 경사하강법$$L[\phi]=\sum_i (\phi_0+\phi_1x_i-y_i)^2,\qquad\frac{\partial \ell_i}{\partial\phi}=\begin{bmatrix}2(\phi_0+\phi_1x_i-y_i)\\ 2x_i(\phi_0+\phi_1x_i-y_i)\end{bmatrix},\qquad\phi\leftarrow\phi-\alpha\frac{\partial L}{\partial\phi}$$

In [ ]:
# 강의 그림과 비슷한 작은 데이터xd = np.array([0.03,0.19,0.34,0.46,0.78,0.81,1.08,1.18,1.39,1.60,1.65,1.90])yd = np.array([0.67,0.85,1.00,1.00,1.40,1.50,1.30,1.54,1.55,1.68,1.73,1.60])def loss(p0, p1):    return np.mean((p0 + p1*xd - yd)**2)def grad(p0, p1, idx=None):    if idx is None: idx = np.arange(len(xd))    e = p0 + p1*xd[idx] - yd[idx]    return 2*e.mean(), 2*(xd[idx]*e).mean()def run_gd(alpha, steps=60, start=(1.6,-0.6)):    p0, p1 = start; path=[(p0,p1)]    for _ in range(steps):        g0,g1 = grad(p0,p1)        p0 -= alpha*g0; p1 -= alpha*g1        path.append((p0,p1))    return np.array(path)# 손실 등고선P0,P1 = np.meshgrid(np.linspace(0,2,200), np.linspace(-1.2,1.6,200))Z = np.array([[loss(a,b) for a in np.linspace(0,2,200)] for b in np.linspace(-1.2,1.6,200)])plt.figure(figsize=(9,3.4))for i, a in enumerate([0.05, 0.3, 0.9]):    path = run_gd(a)    plt.subplot(1,3,i+1)    plt.contourf(P0,P1,Z, levels=30, cmap="copper")    plt.plot(path[:,0], path[:,1], "-o", c="lightcyan", ms=3, lw=1)    plt.title(f"alpha={a}", fontsize=9); plt.xlabel("phi0"); plt.ylabel("phi1")plt.tight_layout(); plt.show()print("작은 alpha → 느린 수렴 / 큰 alpha → 지그재그·발산")print("closed-form 해:", np.round(np.polyfit(xd, yd, 1)[::-1], 4))

---## 7. 확률적 경사하강법 (SGD)$$\phi_{t+1}\leftarrow\phi_t-\alpha\sum_{i\in\mathcal B_t}\frac{\partial\ell_i}{\partial\phi}$$

In [ ]:
def run_sgd(alpha, batch, steps=200, start=(1.6,-0.6), seed=1):    r = np.random.default_rng(seed)    p0,p1 = start; path=[(p0,p1)]; losses=[loss(p0,p1)]    N = len(xd)    for _ in range(steps):        idx = r.choice(N, size=batch, replace=False)        g0,g1 = grad(p0,p1,idx)        p0 -= alpha*g0; p1 -= alpha*g1        path.append((p0,p1)); losses.append(loss(p0,p1))    return np.array(path), np.array(losses)plt.figure(figsize=(9,3.2))plt.subplot(1,2,1)plt.contourf(P0,P1,Z, levels=30, cmap="copper")for batch, c, lab in [(12,"lightcyan","GD (전체 12개)"), (2,"orange","SGD (batch=2)")]:    path,_ = run_sgd(0.1, batch, steps=120)    plt.plot(path[:,0], path[:,1], lw=1, c=c, label=lab)plt.legend(fontsize=8); plt.xlabel("phi0"); plt.ylabel("phi1")plt.subplot(1,2,2)for batch, lab in [(12,"GD"), (4,"SGD b=4"), (1,"SGD b=1")]:    _, ls = run_sgd(0.1, batch, steps=120)    plt.plot(ls, label=lab)plt.yscale("log"); plt.legend(fontsize=8); plt.xlabel("step"); plt.ylabel("loss")plt.tight_layout(); plt.show()print("SGD는 진동(jitter)하지만 스텝당 계산량이 훨씬 적고, 노이즈가 local minima 탈출에 도움")

### 7-1. Non-convex 지형에서 GD vs SGD

In [ ]:
F  = lambda p: 0.12*p**2 + np.sin(2.2*p)*1.3 + 1.6dF = lambda p: 0.24*p + 2.86*np.cos(2.2*p)def descend(start, alpha=0.05, noise=0.0, steps=400, seed=0):    r = np.random.default_rng(seed); p = start; path=[p]    for _ in range(steps):        p = p - alpha*(dF(p) + r.uniform(-noise, noise))        p = np.clip(p, -4, 4); path.append(p)    return np.array(path)ps = np.linspace(-4,4,600)plt.figure(figsize=(6,3.4))plt.plot(ps, F(ps), c="mediumseagreen")for noise, c, lab in [(0.0,"tab:blue","GD"), (1.2,"tab:orange","SGD (noise)")]:    tr = descend(-3.0, noise=noise, seed=7)    plt.plot(tr[::10], F(tr[::10]), "o", ms=3, c=c, label=f"{lab} → phi={tr[-1]:.2f}")plt.legend(fontsize=8); plt.xlabel("phi"); plt.ylabel("Loss")plt.tight_layout(); plt.show()

---## 8. 역전파(Backpropagation) 직접 구현$$\mathbf f_k=\boldsymbol\beta_k+\boldsymbol\Omega_k\mathbf h_k,\quad \mathbf h_{k+1}=\mathrm a[\mathbf f_k]$$$$\frac{\partial\ell}{\partial\boldsymbol\Omega_k}=\frac{\partial\ell}{\partial\mathbf f_k}\mathbf h_k^\top,\qquad\frac{\partial\ell}{\partial\boldsymbol\beta_k}=\frac{\partial\ell}{\partial\mathbf f_k}$$

In [ ]:
class MLP:    """ReLU 은닉층 + 선형 출력, 최소제곱 손실"""    def __init__(self, sizes, seed=0):        r = np.random.default_rng(seed)        self.W = [r.normal(0, np.sqrt(2/sizes[i]), (sizes[i+1], sizes[i])) for i in range(len(sizes)-1)]        self.b = [np.zeros((sizes[i+1],1)) for i in range(len(sizes)-1)]    def forward(self, X):           # X: (Di, N)        self.h = [X]; self.f = []        for k in range(len(self.W)):            fk = self.W[k] @ self.h[-1] + self.b[k]            self.f.append(fk)            self.h.append(relu(fk) if k < len(self.W)-1 else fk)   # 마지막은 활성화 없음        return self.h[-1]    def backward(self, Y):        N = Y.shape[1]        K = len(self.W)        dW = [None]*K; db = [None]*K        # 출력층: l = mean((f_K - y)^2)        dfk = 2*(self.h[-1] - Y)/N            # dl/df_{K-1}        for k in reversed(range(K)):            dW[k] = dfk @ self.h[k].T          # dl/dW_k = dl/df_k · h_k^T            db[k] = dfk.sum(axis=1, keepdims=True)            if k > 0:                dh = self.W[k].T @ dfk         # dl/dh_k                dfk = dh * (self.f[k-1] > 0)   # ReLU 미분 통과        return dW, db    def step(self, dW, db, lr):        for k in range(len(self.W)):            self.W[k] -= lr*dW[k]; self.b[k] -= lr*db[k]def mse(a, b): return np.mean((a-b)**2)

### 8-1. 수치미분으로 역전파 검증 (가장 중요한 습관!)

In [ ]:
net = MLP([1,5,5,1], seed=2)Xg = np.linspace(-1,1,7)[None,:]Yg = np.sin(3*Xg)net.forward(Xg); dW, db = net.backward(Yg)# 수치미분: (L(w+e) - L(w-e)) / 2eeps = 1e-6k, i, j = 1, 2, 3w_save = net.W[k][i,j]net.W[k][i,j] = w_save + eps; Lp = mse(net.forward(Xg), Yg)net.W[k][i,j] = w_save - eps; Lm = mse(net.forward(Xg), Yg)net.W[k][i,j] = w_savenum = (Lp - Lm)/(2*eps)print(f"역전파 gradient  = {dW[k][i,j]: .8f}")print(f"수치미분 gradient = {num: .8f}")print("두 값이 거의 같으면 역전파 구현이 올바릅니다.")

### 8-2. 실제로 학습시켜 보기 — 신경망이 곡선을 배우는 과정

In [ ]:
Xtr = np.linspace(-1, 1, 120)[None,:]Ytr = np.sin(3*Xtr) + 0.05*rng.normal(size=Xtr.shape)net = MLP([1,16,16,1], seed=5)losses = []snap = {}for it in range(4001):    net.forward(Xtr)    dW, db = net.backward(Ytr)    net.step(dW, db, lr=0.05)    if it % 50 == 0: losses.append(mse(net.forward(Xtr), Ytr))    if it in (0, 200, 1000, 4000): snap[it] = net.forward(Xtr).ravel().copy()plt.figure(figsize=(9,3.2))plt.subplot(1,2,1)plt.scatter(Xtr.ravel(), Ytr.ravel(), s=8, c="lightgray")for it, pred in snap.items():    plt.plot(Xtr.ravel(), pred, label=f"iter {it}")plt.legend(fontsize=8); plt.title("학습 과정")plt.subplot(1,2,2)plt.plot(np.arange(len(losses))*50, losses); plt.yscale("log")plt.xlabel("iteration"); plt.ylabel("loss"); plt.title("loss 감소")plt.tight_layout(); plt.show()print("최종 loss =", round(losses[-1], 6))

### 8-3. 미니배치 SGD로 같은 학습을 다시

In [ ]:
net2 = MLP([1,16,16,1], seed=5)r = np.random.default_rng(0)N = Xtr.shape[1]; losses2 = []for it in range(4001):    idx = r.choice(N, 16, replace=False)    net2.forward(Xtr[:,idx]); dW, db = net2.backward(Ytr[:,idx])    net2.step(dW, db, lr=0.05)    if it % 50 == 0: losses2.append(mse(net2.forward(Xtr), Ytr))plt.figure(figsize=(5,3))plt.plot(np.arange(len(losses))*50, losses,  label="full-batch GD")plt.plot(np.arange(len(losses2))*50, losses2, label="mini-batch SGD (16)")plt.yscale("log"); plt.legend(); plt.xlabel("iteration"); plt.ylabel("loss")plt.tight_layout(); plt.show()print("SGD는 진동하지만 스텝당 계산량이 적다 (여기선 데이터가 작아 차이가 작습니다)")

---## 마무리 체크리스트- [ ] 최소제곱 해 $\hat\beta_1=\frac{\sum(x-\bar x)(y-\bar y)}{\sum(x-\bar x)^2}$ 를 직접 코드로 쓸 수 있다- [ ] RSS / TSS / $R^2$ 의 관계를 설명할 수 있다- [ ] 정규방정식 $(\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y$ 를 구현할 수 있다- [ ] 다중공선성이 “예측”이 아니라 “계수 해석”을 망친다는 점을 안다- [ ] 분류에 선형회귀가 부적절한 이유를 그림으로 설명할 수 있다- [ ] 로짓 변환하면 선형이 된다는 것을 수식으로 보일 수 있다- [ ] MLE와 log-likelihood의 관계(단조 변환)를 설명할 수 있다- [ ] ReLU shallow 네트워크의 조각 수 = 유닛 수 + 1 임을 안다- [ ] Deep이 표현력에서 유리한 이유(접기)를 설명할 수 있다- [ ] 경사하강 업데이트식과 학습률의 역할을 안다- [ ] SGD가 계산량·local minima 양쪽에서 유리한 이유를 안다- [ ] 역전파를 수치미분으로 검증할 수 있다**다음:** `02_문제집.ipynb` 로 직접 풀어보세요. 정답은 `03_해답.ipynb`.